<div align="center">
    <span style="font-size:30px">
        <strong>
            <!-- Símbolo de Python -->
            <img
                src="https://cdn3.emoji.gg/emojis/1887_python.png"
                style="margin-bottom:-5px"
                width="30px" 
                height="30px"
            >
            <!-- Título -->
            Python para Geólogos
            <!-- Versión -->
            <img 
                src="https://img.shields.io/github/release/kevinalexandr19/manual-python-geologia.svg?style=flat&label=&color=blue"
                style="margin-bottom:-2px" 
                width="40px"
            >
        </strong>
    </span>
    <br>
    <span>
        <!-- Github del proyecto -->
        <a href="https://github.com/kevinalexandr19/manual-python-geologia" target="_blank">
            <img src="https://img.shields.io/github/stars/kevinalexandr19/manual-python-geologia.svg?style=social&label=Github Repo">
        </a>
        &nbsp;&nbsp;
        <!-- Licencia -->
        <img src="https://img.shields.io/github/license/kevinalexandr19/manual-python-geologia.svg?color=forestgreen">
        &nbsp;&nbsp;
        <!-- Release date -->
        <img src="https://img.shields.io/github/release-date/kevinalexandr19/manual-python-geologia?color=gold">
    </span>
    <br>
    <span>
        <!-- Perfil de LinkedIn -->
        <a target="_blank" href="https://www.linkedin.com/in/kevin-alexander-gomez/">
            <img src="https://img.shields.io/badge/-Kevin Alexander Gomez-5eba00?style=social&logo=linkedin">
        </a>
        &nbsp;&nbsp;
        <!-- Perfil de Github -->
        <a target="_blank" href="https://github.com/kevinalexandr19">
            <img src="https://img.shields.io/github/followers/kevinalexandr19.svg?style=social&label=kevinalexandr19&maxAge=2592000">
        </a>
    </span>
    <br>
</div>

***

<span style="color:lightgreen; font-size:25px">**PG303 - Análisis geoespacial** </span>

Bienvenido al curso!!!

Vamos a revisar algunas aplicaciones del <span style="color:gold">análisis geoespacial</span> usando código en Python. <br>
Es necesario que tengas un conocimiento previo en programación con Python, estadística y sistemas de información geográfica.


<span style="color:gold; font-size:20px">**Rioxarray**</span>

***
- [¿Qué es Rioxarray?](#parte-1)
- [Abrir un ráster en Python](#parte-2)
- [Elevación, NoData y estadística descriptiva](#parte-3)
- [Convertir una ventana ráster a DataFrame](#parte-4)

***

<a id="parte-1"></a>

### <span style="color:lightgreen">**¿Qué es Rioxarray?**</span>
***

<span style="color:gold">Rioxarray</span> agrega operaciones geoespaciales al `DataArray` de Xarray. Un `DataArray` combina valores, dimensiones nombradas y coordenadas. El accesor `.rio` proporciona el CRS, la transformación de píxeles y las herramientas para recortar o reproyectar.

Usaremos una ventana del modelo de elevación del cuaderno original, desde el Pacífico hasta los Andes centrales. La ventana conserva los valores, el CRS y los 60 segundos de arco por píxel de la malla original.

JupyterLite carga Rasterio y Xarray desde el entorno del curso y usa Rioxarray 0.18.2, compatible con Rasterio 1.4.2. En Python local puedes usar `python -m pip install "rioxarray==0.18.2" matplotlib`.

In [ ]:
import sys

if sys.platform == "emscripten":
    import piplite
    from pyodide_js import loadPackage

    await loadPackage(["rasterio", "xarray", "pyproj", "matplotlib"])
    await piplite.install("rioxarray==0.18.2")

<a id="parte-2"></a>

### <span style="color:lightgreen">**Abrir un ráster en Python**</span>
***

Leemos el GeoTIFF pequeño en memoria con `MemoryFile`, de modo que Rasterio pueda acceder a sus bloques dentro del navegador. `open_rasterio` lo abre como un `DataArray`. Usamos `masked=True` para que los píxeles que el archivo marca como NoData aparezcan como `NaN`. `isel(band=0, drop=True)` selecciona la primera banda por su posición y deja las dimensiones `(y, x)`. Aquí `x` es longitud, `y` es latitud y los valores del DEM son elevaciones en metros.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rioxarray as rxr
from rasterio.io import MemoryFile
from rasterio.transform import xy

ruta = Path("files/peru_elevacion_60s.tif")
with MemoryFile(ruta.read_bytes()) as archivo, archivo.open() as fuente:
    raster = rxr.open_rasterio(fuente, masked=True)
    dem = raster.isel(band=0, drop=True).load()

print(f"Dimensiones: {dem.dims} | Forma: {dem.shape} | CRS: {dem.rio.crs}")
print("Resolución (°/píxel):", dem.rio.resolution())
print("NoData codificado en el archivo:", dem.rio.encoded_nodata)
dem

La fila se cuenta desde arriba y la columna desde la izquierda, empezando en cero. Con `isel(y=fila, x=columna)` obtenemos una celda. La transformación afín indica cómo pasar de estos índices a coordenadas en el CRS. En ambos casos veremos el **centro** del píxel. Explica por qué el paso de `y` es negativo cuando avanzamos una fila hacia el sur.

In [ ]:
fila, columna = 100, 150
celda = dem.isel(y=fila, x=columna)
lon, lat = xy(dem.rio.transform(), fila, columna, offset="center")
assert np.allclose([lon, lat], [float(celda.x), float(celda.y)])
print(f"Fila {fila}, columna {columna} → ({lon:.4f}°, {lat:.4f}°) → {float(celda):.1f} m")

<a id="parte-3"></a>

### <span style="color:lightgreen">**Elevación, NoData y estadística descriptiva**</span>
***

Un número como `-9999` podría indicar que falta un dato si está registrado como NoData. Este ejemplo sintético muestra qué pasa con el mínimo al enmascararlo. En el DEM veremos valores negativos reales, asociados a elevaciones bajo el nivel del mar, así que usaremos la máscara del archivo y comprobaremos que los valores sean finitos.

In [ ]:
pixeles = np.array([[120., -9999., 430.], [-12., 130., 240.]])
codigo_nodata = -9999
muestra_enmascarada = np.where(pixeles == codigo_nodata, np.nan, pixeles)
print(f"Mínimo crudo: {pixeles.min():.0f} m | Mínimo válido: {np.nanmin(muestra_enmascarada):.0f} m")

validos = dem.values[np.isfinite(dem.values)]
print(f"Píxeles válidos: {validos.size:,} de {dem.size:,}")
print(f"Mínimo: {validos.min():.1f} m | Máximo: {validos.max():.1f} m")

El mapa muestra **dónde** están los valores. El histograma muestra **cuántos** píxeles caen en cada intervalo. Los negativos incluyen el fondo marino y los positivos incluyen la cordillera. El filtro `dem > 0` permite examinar el terreno elevado. La línea de costa requiere una delimitación propia, ya que la elevación de un píxel es una medida a la resolución del DEM.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 4.1), layout="constrained")
oeste, sur, este, norte = dem.rio.bounds()
imagen = ejes[0].imshow(dem.values, extent=(oeste, este, sur, norte),
                        origin="upper", cmap="terrain",
                        vmin=validos.min(), vmax=validos.max())
ejes[0].scatter(lon, lat, marker="+", s=100, color="black", label="Celda elegida")
ejes[0].set(title="Elevación · Perú central y fondo marino", xlabel="Longitud (°)", ylabel="Latitud (°)")
ejes[0].legend(loc="lower left", fontsize=8)
fig.colorbar(imagen, ax=ejes[0], shrink=.8, label="Elevación (m)")

ejes[1].hist(validos, bins=40, color="#187B82", edgecolor="white", linewidth=.45)
ejes[1].axvline(0, ls="--", color="#A74619", label="0 m")
ejes[1].set(title="Distribución de las celdas válidas", xlabel="Elevación (m)", ylabel="Píxeles")
ejes[1].legend()
ejes[1].grid(axis="y", alpha=.22)
ejes[1].set_axisbelow(True)
plt.show()

In [ ]:
positivos = dem.where(dem > 0)
fig, ax = plt.subplots(figsize=(6, 4.3), layout="constrained")
positivos.plot.imshow(ax=ax, cmap="YlGn", cbar_kwargs={"label": "Elevación (m)"})
ax.set(title="Píxeles con elevación positiva", xlabel="Longitud (°)", ylabel="Latitud (°)")
plt.show()

<a id="parte-4"></a>

### <span style="color:lightgreen">**Convertir una ventana ráster a DataFrame**</span>
***

`isel` recorta una ventana de 20 filas y 30 columnas. `to_dataframe` convierte el rectángulo en una fila por píxel, conservando las coordenadas de cada centro. `reset_index` pasa `x` e `y` del índice a columnas normales. Para reconstruir la malla también necesitamos las dimensiones, el CRS y la transformación original de esa ventana.

In [ ]:
ventana = dem.isel(y=slice(90, 110), x=slice(135, 165))
df = ventana.to_dataframe(name="ELEV").reset_index()
df = df[["x", "y", "ELEV"]].rename(columns={"x": "LON", "y": "LAT"})
assert len(df) == ventana.size
print(f"Malla {ventana.shape} → {len(df)} filas")
df.head()

In [ ]:
ruta_csv = Path("pg303_ventana_elevacion.csv")
ruta_json = Path("pg303_ventana_elevacion.json")
ruta_csv.write_text(df.to_csv(index=False, lineterminator="\n"), encoding="utf-8")
metadatos = {
    "crs": str(ventana.rio.crs),
    "transformacion_afin": list(ventana.rio.transform())[:6],
    "orden_coeficientes": "a, b, c, d, e, f",
    "dimensiones_yx": list(ventana.shape),
    "unidad_elevacion": "m",
}
ruta_json.write_text(json.dumps(metadatos, indent=2), encoding="utf-8")
print(f"Archivos guardados: {ruta_csv} y {ruta_json}")
metadatos

**Comprueba tu lectura.** Busca la celda de la fila 100 y columna 150 en la tabla usando sus dos coordenadas. Descarga el CSV y su JSON de metadatos desde el explorador de archivos. ¿Qué información necesitas para reconstruir la ventana después de ordenar las filas por elevación? Para trabajar con vecinos o recortar por coordenadas, suele ser más directo mantener el `DataArray`.

[CRS y transformaciones en Rioxarray](https://corteva.github.io/rioxarray/stable/getting_started/crs_management.html)